In [12]:
import numpy as np
import pandas as pd

df = pd.read_csv("loan_data.csv")
df = df.dropna()

df = df.sort_values("fico_score").reset_index(drop=True)

df.head()

,customer_id,credit_lines_outstanding,loan_amt_outstanding,total_debt_outstanding,income,years_employed,fico_score,default
0,7264776,1,4457.914800,12233.49501,98913.32028,3,408,0
1,6901345,3,5281.352243,16411.51801,79905.09892,1,409,1
2,2585781,4,6734.984475,26384.58439,97668.03091,2,418,1
3,1252008,5,5176.915602,22990.26543,82417.59227,2,425,1
4,1337395,5,4271.314690,22756.28103,83475.30929,4,438,1


In [13]:
def log_likelihood(n, k):
    if n == 0:
        return -np.inf

    p = k / n
    p = np.clip(p, 1e-6, 1 - 1e-6)

    return k * np.log(p) + (n - k) * np.log(1 - p)


In [14]:
max_score = int(df["fico_score"].max()) + 1

default_prefix = np.zeros(max_score + 1)
count_prefix = np.zeros(max_score + 1)

for i in range(len(df)):
    s = int(df.loc[i, "fico_score"])
    default_prefix[s] += df.loc[i, "default"]
    count_prefix[s] += 1

for i in range(1, len(default_prefix)):
    default_prefix[i] += default_prefix[i - 1]
    count_prefix[i] += count_prefix[i - 1]

In [15]:
def find_best_buckets_dp(df, num_buckets=5):

    R = num_buckets
    N = len(default_prefix)

    dp = np.full((R + 1, N), -1e18)
    split = np.zeros((R + 1, N), dtype=int)

    dp[0][0] = 0

    for r in range(1, R + 1):
        for j in range(1, N):

            for k in range(j):

                n_bucket = count_prefix[j - 1] - count_prefix[k - 1]
                k_bucket = default_prefix[j - 1] - default_prefix[k - 1]

                if n_bucket == 0:
                    continue

                score = dp[r - 1][k] + log_likelihood(n_bucket, k_bucket)

                if score > dp[r][j]:
                    dp[r][j] = score
                    split[r][j] = k

    # backtrack
    bins = []
    j = N - 1

    for r in range(R, 0, -1):
        k = split[r][j]
        bins.append(j)
        j = k

    bins.append(0)

    return sorted(set(bins))

In [16]:
def build_rating_map(df, bins):

    rating_map = {}

    for i in range(len(bins) - 1):

        low, high = bins[i], bins[i + 1]

        bucket = df[(df["fico_score"] >= low) & (df["fico_score"] < high)]

        default_rate = bucket["default"].mean() if len(bucket) > 0 else 0

        rating_map[i + 1] = {
            "range": (low, high),
            "default_rate": default_rate
        }

    return rating_map

In [17]:
def assign_rating(fico, bins):
    for i in range(len(bins) - 1):
        if bins[i] <= fico < bins[i + 1]:
            return i + 1
    return len(bins) - 1

In [19]:
num_buckets = 5

best_bins = find_best_buckets_dp(df, num_buckets=num_buckets)

rating_map = build_rating_map(df, best_bins)

df["fico_rating"] = df["fico_score"].apply(lambda x: assign_rating(x, best_bins))

print("Optimal DP Bins:")
print(best_bins)

Optimal DP Bins:
[0, np.int64(409), np.int64(553), np.int64(611), np.int64(650), 851]


In [20]:
df[["fico_score", "fico_rating", "default"]].head()

,fico_score,fico_rating,default
0,408,1,0
1,409,2,1
2,418,2,1
3,425,2,1
4,438,2,1


In [21]:
print("\nFICO Bucket Summary:")
for i in range(len(best_bins) - 1):
    print(f"Bucket {i+1}: {best_bins[i]} → {best_bins[i+1]}")


FICO Bucket Summary:
Bucket 1: 0 → 409
Bucket 2: 409 → 553
Bucket 3: 553 → 611
Bucket 4: 611 → 650
Bucket 5: 650 → 851
